# Supervised Classification / Regression Modeling & Tuning

**Dataset:** Adult Census Income (OpenML)  
**Task:** Predict whether annual income is above $50K (binary classification).

This notebook trains and compares four distinct classifiers, performs hyperparameter tuning with stratified cross-validation, evaluates test-set performance, plots ROC-AUC curves, selects a champion model using cross-validation ROC-AUC, and saves the fitted model.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

RANDOM_STATE = 42

## 2. Load and prepare the dataset

In [ ]:
dataset = fetch_openml("adult", version=2, as_frame=True)
X = dataset.data.copy()
y = dataset.target.copy()

# Clean missing-value markers and target labels
X = X.replace("?", np.nan)
y = y.astype(str).str.strip().str.rstrip(".")

y = y.map({"<=50K": 0, ">50K": 1})

# Remove any rows with an unmapped target, if present
valid_rows = y.notna()
X = X.loc[valid_rows].copy()
y = y.loc[valid_rows].astype(int)

print("Dataset shape:", X.shape)
print("Target distribution:")
print(y.value_counts())
display(X.head())

## 3. Train/test split before preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

## 4. Build leak-free preprocessing

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    # Dense output supports HistGradientBoostingClassifier.
    sparse_threshold=0
)

print("Preprocessing transformer created.")

## 5. Define four baseline model architectures and tuning grids

In [ ]:
# Each estimator is placed after preprocessing in a single Pipeline.
# Small parameter grids keep the notebook practical to run in Colab.

model_grids = {
    "Logistic Regression": (
        LogisticRegression(max_iter=1500, random_state=RANDOM_STATE),
        {
            "classifier__C": [0.5, 1.0, 2.0],
            "classifier__class_weight": [None, "balanced"]
        }
    ),
    "Random Forest": (
        RandomForestClassifier(
            n_estimators=150,
            random_state=RANDOM_STATE,
            n_jobs=1
        ),
        {
            "classifier__max_depth": [None, 15],
            "classifier__min_samples_leaf": [1, 3]
        }
    ),
    "Extra Trees": (
        ExtraTreesClassifier(
            n_estimators=150,
            random_state=RANDOM_STATE,
            n_jobs=1
        ),
        {
            "classifier__max_depth": [None, 15],
            "classifier__min_samples_leaf": [1, 3]
        }
    ),
    "Hist Gradient Boosting": (
        HistGradientBoostingClassifier(
            random_state=RANDOM_STATE
        ),
        {
            "classifier__learning_rate": [0.05, 0.1],
            "classifier__max_iter": [100, 150],
            "classifier__max_leaf_nodes": [15, 31]
        }
    )
}

print("Number of model architectures:", len(model_grids))

## 6. Hyperparameter optimization with stratified K-fold cross-validation

In [ ]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

searches = {}
comparison_rows = []

for model_name, (estimator, param_grid) in model_grids.items():
    print(f"Training and tuning: {model_name}")

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", estimator)
    ])

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        verbose=1
    )

    search.fit(X_train, y_train)
    searches[model_name] = search

    comparison_rows.append({
        "Model": model_name,
        "Best CV ROC-AUC": search.best_score_,
        "Best Parameters": search.best_params_
    })

model_comparison = pd.DataFrame(comparison_rows).sort_values(
    by="Best CV ROC-AUC",
    ascending=False
).reset_index(drop=True)

display(model_comparison)

## 7. Evaluate tuned models on the held-out test set

In [ ]:
evaluation_rows = []
test_probabilities = {}

for model_name, search in searches.items():
    fitted_model = search.best_estimator_
    predictions = fitted_model.predict(X_test)
    probabilities = fitted_model.predict_proba(X_test)[:, 1]

    test_probabilities[model_name] = probabilities

    evaluation_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })

evaluation_table = pd.DataFrame(evaluation_rows).sort_values(
    by="ROC-AUC",
    ascending=False
).reset_index(drop=True)

display(evaluation_table.round(4))

## 8. Plot ROC-AUC curves

In [ ]:
plt.figure(figsize=(9, 7))

for model_name, probabilities in test_probabilities.items():
    RocCurveDisplay.from_predictions(
        y_test,
        probabilities,
        name=model_name,
        ax=plt.gca()
    )

plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.title("ROC Curves - Tuned Classification Models")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 9. Select the champion model using validation performance

In [ ]:
# Select using the best cross-validation ROC-AUC, not test-set performance.
champion_name = model_comparison.loc[0, "Model"]
champion_search = searches[champion_name]
champion_model = champion_search.best_estimator_

print("Champion model:", champion_name)
print("Best cross-validation ROC-AUC:", round(champion_search.best_score_, 4))
print("Best parameters:", champion_search.best_params_)

## 10. Champion model: classification report and confusion matrix

In [ ]:
champion_predictions = champion_model.predict(X_test)
champion_probabilities = champion_model.predict_proba(X_test)[:, 1]

print(classification_report(
    y_test,
    champion_predictions,
    target_names=["Income <=50K", "Income >50K"],
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    champion_predictions,
    display_labels=["Income <=50K", "Income >50K"],
    cmap="Blues"
)
plt.title(f"Confusion Matrix - {champion_name}")
plt.tight_layout()
plt.show()

print("Champion test ROC-AUC:", round(
    roc_auc_score(y_test, champion_probabilities), 4
))

## 11. Save the champion model

In [ ]:
model_filename = "champion_income_classifier.joblib"
joblib.dump(champion_model, model_filename)

print(f"Champion model saved as: {model_filename}")

## 12. Verify the saved model

In [ ]:
loaded_model = joblib.load(model_filename)
sample_predictions = loaded_model.predict(X_test.iloc[:5])

print("Sample predictions:", sample_predictions.tolist())
print("Model reload verification successful.")

## 13. Conclusion

Four distinct classification architectures were trained and tuned using `GridSearchCV` with stratified 3-fold cross-validation. Preprocessing was included inside each Scikit-learn Pipeline, ensuring that imputation, scaling, and encoding were fitted separately within each training fold.

Models were compared using validation ROC-AUC and evaluated on a held-out test set using accuracy, precision, recall, F1-score, ROC-AUC, classification reports, confusion matrices, and ROC curves. The champion model was selected using the highest cross-validation ROC-AUC and saved as a reusable `.joblib` artifact.

**Submission files:** this notebook (`.ipynb`) and the generated `champion_income_classifier.joblib` file. Run all cells before committing the notebook so the outputs, comparison table, and ROC curves are visible.